<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-12-production-deploy/lesson-12.4-streamlit-frontend/notebooks/GCP_Capstone_12.4_StreamlitFrontend.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.4 Streamlit Frontend — The UI Is a Client
**Netsetos GenAI Engineering — GCP Capstone** · Module 12 · rebuilt on the live lane, 10 September 2026

The user-facing layer, live behind IAP since the first `make up`. This notebook is where `services/frontend` comes from - the heredocs at the end are the kit's source - and the page they build has no model call in it: it streams from the API and renders what it is told. So the lesson is the client's: the notebook reads `/v1/stream` exactly as `chat.py` does and decides each citation's rendering the way `citations.py` decides it; reads the IAP door off the service (and why the API behind it has none); reads the floor a session day sets; and prunes the two pins nothing imported. The one thing a notebook cannot do is sign in through a browser, so the UI's own URL is read as configuration, not clicked.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
UI_URL   = f"https://documind-ui-{NUMBER}.{REGION}.run.app"    # the one surface a browser reaches: IAP is its door
QUESTION = "After how many years of continuous service does gratuity become payable?"

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120, token: str | None = "member") -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) - as documind-ui-sa by default, with token=None as
    nobody, or with a token minted as another account. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    headers = {}
    if token == "member":
        headers["Authorization"] = f"Bearer {documind_tools._id_token(API_URL)}"     # the audience is the canonical URL
    elif token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(f"{url}{path}", json=body, headers=headers, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def api_get(path: str, base: str | None = None, timeout: int = 60) -> tuple[int, dict | str]:
    """GET one API route as the roster member: /version, /health."""
    url = (base or API_URL).rstrip("/")
    r = requests.get(f"{url}{path}", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"), "sa": j["spec"]["template"]["spec"].get("serviceAccountName"),
            "annotations": (j["spec"]["template"]["metadata"].get("annotations") or {}),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query", service_name: str = "documind-api") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="{service_name}" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

# THE TWENTY LINES THAT MATTER, FROM THE CLONE. Module 12's notebooks are where the kit's files come from (the heredoc
# cells at the end of each notebook are what extract_documind.py reads), so the walls stay there and the story reads
# the file the lane actually runs, around one line, with the file's length beside it.
def excerpt(rel: str, needle: str, before: int = 0, after: int = 14) -> str:
    lines = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read().splitlines()
    i = next(n for n, l in enumerate(lines) if needle in l)
    lo, hi = max(0, i - before), min(len(lines), i + after)
    return f"# {rel}:{lo + 1}-{hi}  ({len(lines)} lines)\n" + "\n".join(lines[lo:hi])

def gcloud(*args: str) -> str:
    """One gcloud read, as the notebook's account, stdout only."""
    r = subprocess.run(["gcloud", *args, "--project", PROJECT_ID], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else f"(gcloud: {r.stderr.strip()[:200]})"

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # run_eval, usage_rows, judge
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules, for the excerpts and the pure functions
print("helpers: api(), api_get(), gateway(), service(), usage_rows(), excerpt(), gcloud(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: The UI is a client
chat.py's read loop and citations.py's rendering decision, from a notebook.


In [ ]:
# THE UI IS A CLIENT. chat.py has no model call in it and never will: it POSTs /v1/stream and renders what it is told.
# This cell is chat.py's read loop in shape - citations first, then tokens, then done - and citations.py's decision
# for each citation: a text chunk opens its page, a figure renders inline, a segment starts the video at its second
# (9.6). What the UI shows is what the API sent; 8.7's seam - the thing that decides separate from the thing that
# displays - is why a retrieval or model change never needs a UI deploy.
print(excerpt("services/frontend/chat.py", "/v1/stream", 2, 16)); print()
body = {"query": QUESTION, "tenant_id": TENANT, "user_id": "u_12_4", "top_k": 5, "stream": True}
r = requests.post(f"{API_URL}/v1/stream", json=body, headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, stream=True, timeout=120)
assert r.status_code == 200, (r.status_code, r.text[:200])
cites, tokens, done, ev = [], [], None, None
for line in r.iter_lines(decode_unicode=True):
    if line.startswith("event: "):
        ev = line[7:]
    elif line.startswith("data: ") and ev:
        payload = json.loads(line[6:])
        if ev == "citation":
            cites.append(payload)
        elif ev == "token":
            tokens.append(payload["t"])
        elif ev == "done":
            done = payload
print(f"{len(cites)} citations, {len(tokens)} tokens, {done['latency_ms']} ms, {done.get('model')} via {done.get('backend')}")
for c in cites[:5]:
    kind = c.get("kind") or "text"
    if kind == "figure":
        render = f"pill [{c['n']}] -> the figure inline, captioned by page: {c.get('media_url')}"
    elif kind == "segment":
        render = f"pill [{c['n']}] -> the video from {c.get('start')}s to {c.get('end')}s: {c.get('media_url')}"
    else:
        render = f"pill [{c['n']}] -> a V4 signed URL of {c['source'].split('/')[-1]}#page={c.get('page')}"
    print(f"  {kind:8} {render}")
print("\nanswer:", "".join(tokens)[:150].replace(chr(10), " "), "...")
assert cites and tokens and done


## Cell 3: The door is IAP


In [ ]:
# THE DOOR IS IAP. The UI is the one surface a browser reaches, so IAP fronts it: --iap on the service, an audience that
# names THIS service, and the humans granted iap.httpsResourceAccessor one by one (ADMIN_EMAILS). Behind it the API has
# no IAP at all - nothing browses it; the UI calls it with its own ID token and forwards the person's assertion inside
# the same request (12.8's two legs), and the API's IAP_AUDIENCE therefore names the UI's audience, not its own. Read
# from the services, never assumed - the four ways to get the audience wrong all present as a permanent 401.
ui = service("documind-ui")
assert ui, "documind-ui is not deployed: make deploy-services (commands/lesson-12.4.sh)"
print("env   :", {k: ui["env"].get(k) for k in ("AUTH_MODE", "IAP_AUDIENCE", "RAG_API_URL", "CHAT_URL", "ADMIN_EMAILS")})
meta = json.loads(gcloud("run", "services", "describe", "documind-ui", "--region", REGION, "--format=json") or "{}")
iap = {k: v for k, v in ((meta.get("metadata") or {}).get("annotations") or {}).items() if "iap" in k.lower()}
print("iap   :", iap or "(no iap annotation on the service metadata - read the deploy script: --iap)")
print("floor :", ui["min_instances"], "(MIN_INSTANCES: 1 on a session day, 0 after; documind-off resets it at 23:00)")
assert ui["env"].get("AUTH_MODE") == "iap" and ui["env"].get("IAP_AUDIENCE", "").endswith("/services/documind-ui"), ui["env"].get("IAP_AUDIENCE")
pol = gcloud("iap", "web", "get-iam-policy", "--resource-type=cloud-run", "--service=documind-ui", f"--region={REGION}", "--format=json")
try:
    print("humans:", [m for b in json.loads(pol).get("bindings", []) for m in b.get("members", [])])
except ValueError:
    print("humans:", pol[:160])
api_env = service("documind-api")["env"]
print("the API: IAP_AUDIENCE =", api_env.get("IAP_AUDIENCE"))
assert "/services/documind-ui" in api_env.get("IAP_AUDIENCE", ""), "the API verifies the assertion IAP minted for the UI, so its audience list names the UI"


## Cell 4: The prune, and the files


In [ ]:
import glob as _glob

# THE PRUNE, AND THE FILES. The frontend's requirements carried litellm and openai from a first draft that generated in
# the UI; no file under services/frontend imports either, and every pin is weight on the one service a person waits
# for at a cold start. Gone (12.4's REQUIREMENTS, readopted). The file table is read from the directory, not typed:
# the first version listed rag.py, deleted two cells earlier in the same notebook.
req = open(f"{KIT}/deploy/services/frontend/requirements.txt", encoding="utf-8").read()
assert "litellm" not in req and "openai" not in req, "the frontend ships no model client"
files = sorted(os.path.basename(f) for f in _glob.glob(f"{KIT}/deploy/services/frontend/*.py"))
print("files:", files, "| pins:", len([l for l in req.splitlines() if "==" in l]))
for f in files:
    src = open(f"{KIT}/deploy/services/frontend/{f}", encoding="utf-8").read()
    assert "litellm" not in src and "import openai" not in src, f"{f} imports a model client"
    doc = src.split(chr(34) * 3)[1].strip().splitlines()[0] if src.count(chr(34) * 3) >= 2 else ""
    print(f"  {f:20} {len(src.splitlines()):>4} lines  {doc[:72]}")
assert "rag.py" not in files, "rag.py was deleted: two retrieval stacks is one too many"
print("\nthe ten checks the smokes run, instead of a 22-item manual list: make smoke-all (12.8)")


## Where this goes
- **12.8** puts the second human surface (chat) behind the same verifier and runs the smoke that proves both doors refuse.
- **12.6**'s guard changes what the stream carries when it is on: tokens held and screened whole, then sent, or an error event.

## ✅ Lesson 12.4 complete
- ✅ /v1/stream read as chat.py reads it; each citation's rendering decided as citations.py decides it
- ✅ IAP read off the service: the mode, the audience, the accessors; the API's audience list names the UI
- ✅ The floor on a session day, and what resets it
- ✅ Two pins nothing imports, gone; the file table read from the directory


## The files this lesson owns
Below are the twelve heredocs the extractor turns into `deploy/services/frontend/*` and `deploy/commands/lesson-12.4.sh`: the requirements (pruned tonight, through `readopt`), the entrypoint, the chat page, the documents page, voice, the Studio, identity, citations, the admin link, Streamlit's config, the Dockerfile and the deploy with IAP. Unchanged by the rebuild otherwise; the story above reads them from the clone.


In [ ]:
REQUIREMENTS = '''
streamlit==1.63.0
Authlib==1.6.6
google-cloud-aiplatform==1.153.1
google-cloud-discoveryengine==0.13.11
google-cloud-storage==3.13.1
google-cloud-documentai==3.15.0
google-cloud-firestore==2.30.0
google-cloud-bigquery[pandas]==3.45.0
google-cloud-secret-manager==2.30.0
google-cloud-speech==2.40.0
google-cloud-texttospeech==2.37.0
google-cloud-dlp==3.39.0
google-genai==2.22.0
langchain-text-splitters==1.1.2
streamlit-pdf-viewer==0.0.28
streamlit-mic-recorder==0.0.8
streamlit-cookies-manager==0.2.0
PyJWT[crypto]==2.13.0
tiktoken==0.14.0
tenacity==9.1.4
fpdf2==2.8.8
plotly==5.24.1
pandas==2.3.3
'''
with open('requirements.txt', 'w') as f:
    f.write(REQUIREMENTS)
print('requirements.txt written with pinned versions for April 2026')


In [ ]:
APP_PY = '''
import os
import streamlit as st
from auth import login_gate, is_admin
from chat import chat_page
from documents import documents_page
from studio import studio_page
from admin_dashboard import admin_page

st.set_page_config(page_title="DocuMind", page_icon="📄", layout="wide",
                   initial_sidebar_state="expanded")

# Auth gate - st.login (dev) or IAP JWT (prod)
user = login_gate()

# Navigation - admin tab only visible to admins. Studio (9.4) is for every roster member:
# the API refuses a tenant the caller is not on, so the tab needs no gate of its own.
pages = ["Chat", "Documents", "Studio", "Admin"] if is_admin(user) else ["Chat", "Documents", "Studio"]
with st.sidebar:
    st.markdown(f"### 👤 {user.get('email')}")
    if st.button("Sign out"):
        st.logout() if os.getenv("AUTH_MODE") != "iap" else st.markdown("Close tab to sign out of IAP")
    st.divider()
    page = st.radio("Navigation", pages, label_visibility="collapsed")

if page == "Chat":
    chat_page(user)
elif page == "Documents":
    documents_page(user)
elif page == "Studio":
    studio_page(user)
elif page == "Admin":
    admin_page(user)
'''
# app.py imports admin_dashboard (Cell 9); chat.py pulls in rag (Cell 7) and
# citations (Cell 8) — all written below, so `streamlit run app.py` resolves cleanly.
with open('app.py', 'w') as f:
    f.write(APP_PY)
print('app.py written')


In [ ]:
CHAT_PY = '''
"""DocuMind chat - a thin client over rag-api.

There is no model call in this file and there must never be one. Retrieval,
grounding, citations, the prompt and the cost accounting all live in rag-api;
the UI streams what it is told and renders it. The version this replaced
generated through LiteLLM, imported answer_query and render_with_citations,
and used neither - a plain chatbot wearing a RAG product's name.
"""
import json
import os
import uuid

import requests
import streamlit as st
import google.auth.transport.requests
import google.oauth2.id_token

from citations import render_with_citations
from auth import tenant_for
from voice import cached_tts, transcribe

RAG_API_URL = os.environ["RAG_API_URL"]
# The chat SERVICE (12.8). When set, the page reaches it: three agent brains over the one
# retrieve() (8.7, gap G6), chosen per turn from the sidebar, plus a link to open the surface
# itself (gap G13). Unset - the local profile, or no chat deployed - and this page stays what
# it always was: a thin streaming client over rag-api, which is the "direct" brain.
CHAT_URL = os.environ.get("CHAT_URL", "").rstrip("/")
BRAINS = ("direct", "langchain", "langgraph", "adk")
USD_INR = 85
# gemini-3.6-flash standard rates, USD per 1M tokens. The $0.75/$3.75
# introductory price runs to 31 Dec 2026.
PRICE_IN, PRICE_OUT = 1.50, 7.50


def _id_token(audience: str) -> str:
    """Minted per call - a Cloud Run ID token lasts an hour (lesson 7.3)."""
    return google.oauth2.id_token.fetch_id_token(
        google.auth.transport.requests.Request(), audience)


def _headers(audience: str = RAG_API_URL) -> dict:
    """TWO credentials, two different gates.

    Authorization: the frontend's own ID token, audience = the service URL being called
    (rag-api, or the chat service). This gets past Cloud Run IAM, and is why ui-sa needs
    roles/run.invoker.

    x-goog-iap-jwt-assertion: the END USER's assertion, forwarded unchanged so the
    callee's shared/iap.py sees the person rather than this service account. Without
    it every request would look like it came from the frontend, and per-user
    audit and per-tenant membership would both be meaningless.
    """
    h = {"Authorization": f"Bearer {_id_token(audience)}"}
    assertion = (st.context.headers or {}).get("x-goog-iap-jwt-assertion")
    if assertion:
        h["x-goog-iap-jwt-assertion"] = assertion
    return h


def stream_answer(query: str, tenant_id: str):
    """Yield (event_name, payload) from rag-api's SSE: citation -> token -> done."""
    with requests.post(
        f"{RAG_API_URL}/v1/stream",
        json={"query": query, "tenant_id": tenant_id, "top_k": 5, "brain": "ui"},
        headers=_headers(), stream=True, timeout=120,
    ) as resp:
        resp.raise_for_status()
        event = None
        for line in resp.iter_lines(decode_unicode=True):
            if not line:
                continue
            if line.startswith("event: "):
                event = line[7:].strip()
            elif line.startswith("data: "):
                yield event, json.loads(line[6:])


def chat_page(user):
    tenant_id = tenant_for(user["email"])
    if not tenant_id:
        st.error("Your account is not a member of any DocuMind tenant. "
                 "Ask an administrator to add you.")
        st.stop()

    with st.sidebar:
        st.caption(f"Tenant: **{tenant_id}**")
        if "session_id" not in st.session_state:
            st.session_state.session_id = uuid.uuid4().hex[:12]   # one conversation per browser session
        if CHAT_URL:
            # 8.7 in production (gap G6): the same question, any of the three agent brains or
            # the direct path, all over the one retrieve(). The chat service logs which one.
            brain = st.radio("Brain", BRAINS, index=0,
                             help="direct = this page streaming from rag-api; the others run "
                                  "in documind-chat with tools, a guard and a checkpointer")
            st.link_button("Open DocuMind Chat", CHAT_URL)   # the surface itself (gap G13)
        else:
            brain = "direct"
        # No temperature or top_p. gemini-3.6-flash ignores temperature, top_p
        # and top_k, so a slider here would be a control that does nothing -
        # worse than no control, because people tune it and believe the result.
        # 9.2, wired: Chirp 3 HD reads the answer back, cached in the TTS bucket by the SHA-256
        # of voice and text, so the second time a figure is read out costs nothing.
        st.session_state.read_aloud = st.toggle("Read answers aloud", value=bool(st.session_state.get("read_aloud")),
                                                help="Chirp 3 HD (en-IN) via voice.cached_tts; free on a cache hit")
        st.divider()
        st.subheader("Cost this session")
        ss = st.session_state
        st.metric("USD", f"${ss.get('session_cost_usd', 0):.4f}",
                  delta=f"+${ss.get('last_turn_cost', 0):.4f} last turn")
        st.metric("INR", f"Rs {ss.get('session_cost_usd', 0) * USD_INR:.2f}")
        st.metric("Tokens", f"{ss.get('tokens_in', 0)} in / "
                            f"{ss.get('tokens_out', 0)} out")

    st.title("DocuMind Chat")
    if "messages" not in st.session_state:
        st.session_state.messages = []
    for m in st.session_state.messages:
        with st.chat_message(m["role"]):
            st.markdown(m["content"])

    spoken = transcribe_from_mic()
    typed = st.chat_input("Ask DocuMind...", max_chars=4000)
    prompt = spoken or typed
    if not prompt:
        return

    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    if brain != "direct":
        # An AGENT brain: the chat service runs the loop (tools, guard, checkpointer) and this
        # page renders what it returns. session_id keeps one conversation per browser session,
        # inside this user's own tenant - the service prefixes both from the verified identity.
        with st.chat_message("assistant"):
            with st.spinner(f"{brain} is thinking..."):
                r = requests.post(f"{CHAT_URL}/v1/chat",
                                  json={"question": prompt, "session_id": st.session_state.session_id,
                                        "brain": brain},
                                  headers=_headers(CHAT_URL), timeout=120)
            if r.status_code != 200:
                st.error(f"chat service returned {r.status_code}: {r.text[:200]}")
                return
            body = r.json()
            answer = body.get("answer", "")
            answer = answer if isinstance(answer, str) else str(answer)
            st.markdown(answer)
            if st.session_state.get("read_aloud") and answer:
                st.audio(cached_tts(answer[:1500]), format="audio/ogg")
            st.caption(f"brain: {body.get('brain')} · tools: {', '.join(body.get('tool_calls') or []) or 'none'}"
                       + (f" · refused: {', '.join(body['refusals'])}" if body.get("refusals") else "")
                       + f" · {body.get('latency_ms', 0)} ms")
        st.session_state.messages.append({"role": "assistant", "content": answer})
        return

    with st.chat_message("assistant"):
        sources, answer, done = [], "", {}
        slot = st.empty()
        for event, payload in stream_answer(prompt, tenant_id):
            if event == "citation":
                # Citations arrive BEFORE the first token - they come from
                # retrieval, so the sources render while the answer is written.
                # The five fields a citation always had, plus 9.6's four: a figure or a
                # video segment arrives here with a locator, and citations.py renders it.
                # Drop these keys and a figure comes back as plain text with no thumbnail,
                # nothing raised, nothing logged - the silent projection 9.6 warned about.
                sources.append({"text": payload.get("quote", ""),
                                "source_uri": payload["source"],
                                "page_start": payload.get("page"),
                                "kind": payload.get("kind", "text"),
                                "media_url": payload.get("media_url"),
                                "start": payload.get("start"),
                                "end": payload.get("end"),
                                "effective_from": payload.get("effective_from")})
            elif event == "token":
                answer += payload["t"]
                slot.markdown(answer)
            elif event == "done":
                done = payload
        slot.empty()
        render_with_citations(answer, sources)
        if st.session_state.get("read_aloud") and answer:
            st.audio(cached_tts(answer[:1500]), format="audio/ogg")

    cost = (done.get("tokens_in", 0) * PRICE_IN
            + done.get("tokens_out", 0) * PRICE_OUT) / 1_000_000
    ss = st.session_state
    ss.session_cost_usd = ss.get("session_cost_usd", 0) + cost
    ss.last_turn_cost = cost
    ss.tokens_in = ss.get("tokens_in", 0) + done.get("tokens_in", 0)
    ss.tokens_out = ss.get("tokens_out", 0) + done.get("tokens_out", 0)
    st.session_state.messages.append({"role": "assistant", "content": answer})


def transcribe_from_mic() -> str:
    """voice.py, wired. It shipped in the image and no page imported it."""
    try:
        from streamlit_mic_recorder import mic_recorder
    except ImportError:
        return ""
    audio = mic_recorder(start_prompt="Speak", stop_prompt="Stop",
                         just_once=True, key="documind-mic")
    if not audio or not audio.get("bytes"):
        return ""
    return transcribe(audio["bytes"])
'''

with open('chat.py', 'w') as f: f.write(CHAT_PY)
print('chat.py written - a thin client over rag-api, no model call in it')


In [ ]:
DOCS_PY = '''
import os
import requests
import streamlit as st
from auth import tenant_for
from chat import RAG_API_URL, _headers
from google.cloud import storage, documentai_v1 as docai
from google import genai
from google.genai import types
from google.cloud import aiplatform

_storage = storage.Client()
BUCKET = _storage.bucket(os.environ["UPLOAD_BUCKET"])
_docai = docai.DocumentProcessorServiceClient(
    client_options={"api_endpoint": "us-documentai.googleapis.com"})
_genai = genai.Client(enterprise=True, project=os.environ.get("GOOGLE_CLOUD_PROJECT"), location=os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1"))
LAYOUT_PROCESSOR = os.environ["LAYOUT_PROCESSOR"]

def extract_chunks(doc, source_uri):
    chunks = []
    for i, c in enumerate(doc.chunked_document.chunks):
        chunks.append({
            "id": f"{source_uri}#{i}",
            "text": c.content,
            "page_start": c.page_span.page_start if c.page_span else None,
            "source_uri": source_uri,
        })
    return chunks

def parse_layout(gcs_uri):
    req = docai.ProcessRequest(
        name=f"{LAYOUT_PROCESSOR}/processorVersions/pretrained-layout-parser-v1.5-2025-08-25",
        gcs_document=docai.GcsDocument(gcs_uri=gcs_uri, mime_type="application/pdf"),
        process_options=docai.ProcessOptions(
            layout_config=docai.ProcessOptions.LayoutConfig(
                chunking_config=docai.ProcessOptions.LayoutConfig.ChunkingConfig(
                    chunk_size=500, include_ancestor_headings=True))))
    return _docai.process_document(request=req).document

def embed_batch(chunks):
    for i in range(0, len(chunks), 5):
        batch = chunks[i:i+5]
        resp = _genai.models.embed_content(
            model="text-embedding-005",
            contents=[c["text"] for c in batch],
            config=types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT", output_dimensionality=768))
        for c, e in zip(batch, resp.embeddings):
            c["embedding"] = e.values
    return chunks

def _held_in(row: dict) -> str:
    """Where a version is held: the managed stores that confirmed it, with the region each holds it in (`mirrored`, the
    worker's stamp on the ledger row) - or the kit's own rows alone, which is every version of an `in` tenant."""
    held = row.get("mirrored") or {}
    return ", ".join(f"{store} ({region})" for store, region in sorted(held.items())) or "kit rows"


def versions_section(tenant_id: str) -> None:
    """The versions view (12 September 2026): GET /v1/sources - the tenant's ledger, as the API serves it. Which
    version of every document is current, what the last reindex cost (chunks reused by hash, embedded, retired),
    the date a document declares, when it landed - and, since 13 September 2026 (evening), where each version is HELD:
    the managed stores that confirmed it (the worker's mirror stamps `mirrored` on the ledger row) beside the tenant's
    data_region, the policy those copies were judged under. Rendered, never queried here: the page holds no Firestore
    credential for the ledger, the API checks the roster, and the same rows are `make sources TENANT=`."""
    st.subheader("Versions")
    try:
        r = requests.get(f"{RAG_API_URL}/v1/sources", params={"tenant_id": tenant_id}, headers=_headers(), timeout=30)
    except Exception as e:  # noqa: BLE001 - the ledger is a view; the upload path above it must not break
        st.info(f"The ledger is not reachable right now ({type(e).__name__}).")
        return
    if r.status_code == 404:
        st.info("This API revision predates the ledger's versions view (GET /v1/sources).")
        return
    if r.status_code != 200:
        st.info(f"The ledger answered {r.status_code}.")
        return
    body = r.json()
    rows = body.get("sources") or []
    if not rows:
        st.info("No documents indexed for this tenant yet.")
        return
    region = body.get("data_region") or "in"
    st.caption(f"{body.get('versions') or len(rows)} current versions - corpus fingerprint {body.get('fingerprint') or 'none yet'}"
               f" (last change: {body.get('last_event') or '-'}) - data_region {region}: "
               + ("the managed stores may hold copies" if region == "any" else "the kit's own rows only"))
    st.dataframe([{"document": r_["name"], "status": r_["status"], "chunks": r_["chunks"], "reused": r_["reused"],
                   "embedded": r_["embedded"], "retired": r_["retired"], "effective from": r_["effective_from"] or "",
                   "embedding": r_["embedding"], "indexed": (r_["indexed_at"] or "")[:19].replace("T", " "),
                   "held in": _held_in(r_)}
                  for r_ in rows], use_container_width=True)
    st.caption("A re-issued document keeps its name: the worker retires the previous version (never deletes it), reuses "
               "every chunk whose text did not change, and the retired rows expire by policy after the retention window.")


def documents_page(user):
    st.title("📄 Documents")
    # The tenant ONCE, before anything is rendered or stored, and a stop when there is none - the
    # gate chat.py and studio.py already have. Until 12 September 2026 this page asked tenant_for()
    # twice and never looked at the answer: a signed-in person on no roster filed every document
    # under "None/<file>", and the ingest worker indexed a tenant called None (12.5, contracts.py).
    # IAP says who you are; only the roster says where your documents go.
    tenant_id = tenant_for(user["email"])
    if not tenant_id:
        st.error("Your account is not a member of any DocuMind tenant. "
                 "Ask an administrator to add you.")
        st.stop()
    versions_section(tenant_id)
    # Documents AND media (9.4 / 9.6): an image, a video or a recording is a document to the
    # ingest worker - it is described, not parsed, and its caption or segments join the same
    # chunks the text does. The list is what the worker's MEDIA_TYPES and parser accept.
    files = st.file_uploader("Upload documents for indexing",
                             type=["pdf", "docx", "txt", "md", "png", "jpg", "jpeg", "mp4", "mp3"],
                             accept_multiple_files=True,
                             max_upload_size=200)

    if files and st.button("Index documents"):
        with st.status("Processing...", expanded=True) as status:
            for f in files:
                st.write(f"📤 Uploading {f.name}")
                # Tenant, not user. Documents belong to the company that owns them,
                # and keying on `sub` filed every colleague's copy separately -
                # which is why nothing uploaded here was ever findable by anyone else.
                #
                # `{tenant}/{name}`: the ingest worker reads the tenant from the FIRST path
                # segment of the object it is told about (12.5, contracts.py), exactly as
                # evals/upload.sh names the corpus. The earlier `tenants/{tenant}/{id}/`
                # shape filed every upload from this page under a tenant called "tenants".
                blob = BUCKET.blob(f"{tenant_id}/{f.name}")
                blob.chunk_size = 8 * 1024 * 1024
                # content_type is what the worker keys its media branch on - the notification
                # carries it; the extension is never consulted.
                blob.upload_from_file(f, content_type=f.type, timeout=300)
                gcs_uri = f"gs://{BUCKET.name}/{blob.name}"
                if (f.type or "").split("/")[0] in ("image", "video", "audio"):
                    st.write(f"🎞️ {f.name} ({f.type}) handed to documind-ingest: described by Gemini, "
                             f"scanned, indexed as a figure or segments. list_documents (MCP) shows it.")
                    continue

                st.write(f"🔍 Parsing layout ({f.name})")
                doc = parse_layout(gcs_uri)
                chunks = extract_chunks(doc, gcs_uri)

                st.write(f"📈 Embedding {len(chunks)} chunks")
                chunks = embed_batch(chunks)

                st.write(f"💾 Upserting to Vector Search")
                # upsert_datapoints(chunks, user["sub"])  # Module 11 pattern

            status.update(label="Done!", state="complete")
'''
with open('documents.py', 'w') as f:
    f.write(DOCS_PY)
print('documents.py written')


In [ ]:
VOICE_PY = '''
import os, hashlib
import streamlit as st
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech as cs
from google.api_core.client_options import ClientOptions
from google.cloud import texttospeech as tts, storage

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
REGION = os.environ.get("SPEECH_REGION", "asia-south1")
TTS_CACHE_BUCKET = storage.Client().bucket(os.environ["TTS_CACHE_BUCKET"])

_speech = SpeechClient(client_options=ClientOptions(
    api_endpoint=f"{REGION}-speech.googleapis.com"))
_tts = tts.TextToSpeechClient()

def transcribe(audio_bytes, language_codes=("hi-IN", "en-IN"), model="chirp_3"):
    cfg = cs.RecognitionConfig(
        auto_decoding_config=cs.AutoDetectDecodingConfig(),
        language_codes=list(language_codes),
        model=model,
        features=cs.RecognitionFeatures(enable_automatic_punctuation=True))
    req = cs.RecognizeRequest(
        recognizer=f"projects/{PROJECT}/locations/{REGION}/recognizers/_",
        config=cfg, content=audio_bytes)
    resp = None
    try: resp = _speech.recognize(request=req)
    except Exception:
        for m in ("chirp_2", "long"):
            cfg.model = m; req.config = cfg
            try: resp = _speech.recognize(request=req); break
            except Exception: continue
    if resp is None:
        return ""   # all STT attempts failed; degrade gracefully
    return " ".join(r.alternatives[0].transcript for r in resp.results
                    if r.alternatives).strip()

def cached_tts(text, voice="en-IN-Chirp3-HD-Kore", lang="en-IN", rate=1.0):
    cache_key = hashlib.sha256(f"{voice}|{rate}|ogg|{text}".encode()).hexdigest()
    blob = TTS_CACHE_BUCKET.blob(f"tts/{cache_key}.ogg")
    if blob.exists():
        return blob.download_as_bytes()
    # Cache miss - synthesize
    cfg = tts.StreamingSynthesizeConfig(
        voice=tts.VoiceSelectionParams(language_code=lang, name=voice),
        streaming_audio_config=tts.StreamingAudioConfig(
            audio_encoding=tts.AudioEncoding.OGG_OPUS, speaking_rate=rate))
    def gen():
        yield tts.StreamingSynthesizeRequest(streaming_config=cfg)
        yield tts.StreamingSynthesizeRequest(
            input=tts.StreamingSynthesisInput(text=text))
    audio = b"".join(r.audio_content for r in _tts.streaming_synthesize(gen()))
    blob.upload_from_string(audio, content_type="audio/ogg")
    return audio
'''
with open('voice.py', 'w') as f:
    f.write(VOICE_PY)
print('voice.py written')
print()
print('Chirp 3 STT: language_codes=["auto"] for Hinglish code-switching')
print('Chirp 3: HD TTS: 30 named speakers, $30/1M chars, SHA-256 GCS cache')
print('Cache hit: 1.2s -> 80ms AND $0 TTS cost on repeats')


In [ ]:
STUDIO_PY = '''
"""The Media Studio tab - lesson 9.4's client half, over the API's /v1/media routes (gap: the
routes shipped on 5 September and nothing called them until Module 9 joined the lane).

Two things, and nothing that the API does not already guard: a prompt becomes an image through
/v1/media/generate - the API checks the tenant's roster, spends once, caches under DEMO_MODE,
writes the audit row and the usage row - and an answer becomes speech through voice.cached_tts
(9.2: Chirp 3 HD, cached in the TTS bucket by the SHA-256 of voice + text). This page never
holds a model client of its own; it signs the generated blob's URL the way citations.py signs a
figure's, and shows it.
"""
import os
import requests
import streamlit as st

from auth import tenant_for
from chat import RAG_API_URL, _headers
from citations import signed_url
from voice import cached_tts

IMAGE_USD, USD_INR = 0.039, 85
DEFAULT_PROMPT = ("A clean, professional grouped bar chart titled 'ACME revenue by region, FY2025 vs FY2026 "
                  "(Rs crore)': India 412 to 508, APAC 188 to 236, EMEA 96 to 91, Americas 143 to 170. "
                  "Teal palette, value labels on every bar, white background.")


def studio_page(user):
    tenant_id = tenant_for(user["email"])
    if not tenant_id:
        st.error("Your account is not a member of any DocuMind tenant.")
        st.stop()
    st.title("Media Studio")
    st.caption(f"Tenant: **{tenant_id}** - every generation is roster-checked, audited (prompt hashed) "
               f"and metered by the API; the image carries SynthID.")

    st.subheader("Generate an image")
    prompt = st.text_area("Prompt", DEFAULT_PROMPT, height=110, max_chars=2000)
    if st.button("Generate", type="primary"):
        with st.spinner("generating..."):
            r = requests.post(f"{RAG_API_URL}/v1/media/generate",
                              json={"prompt": prompt, "tenant_id": tenant_id},
                              headers=_headers(), timeout=120)
        if r.status_code != 200:
            st.error(f"media route returned {r.status_code}: {r.text[:300]}")
        else:
            body = r.json()
            bucket = body.get("bucket") or os.environ.get("MEDIA_BUCKET", f"{os.environ['GOOGLE_CLOUD_PROJECT']}-media")
            st.image(signed_url(f"gs://{bucket}/{body['blob']}"), caption=body["blob"])
            cost = 0.0 if body.get("cached") else IMAGE_USD
            st.caption(f"{'cache hit (DEMO_MODE)' if body.get('cached') else 'generated'} - "
                       f"${cost:.3f} / Rs {cost * USD_INR:.2f} - {body['blob']}")

    st.divider()
    st.subheader("Read it aloud")
    text = st.text_area("Text", "EMEA revenue fell 5.2 per cent, from 96 crore to 91 crore.", height=80, max_chars=1500)
    voice = st.selectbox("Voice", ["en-IN-Chirp3-HD-Kore", "en-IN-Chirp3-HD-Charon", "hi-IN-Chirp3-HD-Kore"])
    if st.button("Speak"):
        lang = voice[:5]
        with st.spinner("synthesising..."):
            audio = cached_tts(text, voice=voice, lang=lang)
        st.audio(audio, format="audio/ogg")
        st.caption("Chirp 3 HD, cached by SHA-256 of voice and text in the TTS bucket: the second play is free.")
'''

with open('studio.py', 'w') as f:
    f.write(STUDIO_PY)
print('studio.py written - 9.4 over the API; the API guards, this page shows')


In [ ]:
AUTH_PY = '''
import os
import streamlit as st
import jwt
from jwt import PyJWKClient

IAP_AUDIENCE = os.getenv("IAP_AUDIENCE")
ADMIN_EMAILS = set(os.getenv("ADMIN_EMAILS", "").split(","))
ADMIN_DOMAINS = set(os.getenv("ADMIN_DOMAINS", "").split(","))
_JWKS = PyJWKClient("https://www.gstatic.com/iap/verify/public_key-jwk")

def _verify_iap_jwt(token, audience):
    key = _JWKS.get_signing_key_from_jwt(token).key
    return jwt.decode(token, key, algorithms=["ES256"],
                      audience=audience, issuer="https://cloud.google.com/iap")

def current_user():
    if os.getenv("AUTH_MODE") == "iap":
        h = st.context.headers or {}
        token = h.get("x-goog-iap-jwt-assertion")
        if not token:
            return {"is_logged_in": False}
        try:
            claims = _verify_iap_jwt(token, IAP_AUDIENCE)
            return {"email": claims["email"], "sub": claims["sub"],
                    "is_logged_in": True}
        except Exception as e:
            st.error(f"IAP JWT verification failed: {e}")
            return {"is_logged_in": False}
    if st.user.is_logged_in:
        return {"email": st.user.email, "sub": st.user.sub,
                "name": getattr(st.user, "name", ""), "is_logged_in": True}
    return {"is_logged_in": False}

def login_gate():
    u = current_user()
    if not u["is_logged_in"]:
        st.title("🔐 DocuMind - Sign in required")
        if os.getenv("AUTH_MODE") != "iap":
            st.button("Log in with Google", on_click=st.login)
        else:
            st.error("IAP authentication missing. Contact admin.")
        st.stop()
    return u

def tenant_for(email: str) -> str | None:
    """Which tenant does this person belong to?

    NOT user["sub"]. The version this replaced used the Google subject id as
    the tenant, which quietly made every user their own tenant: two colleagues
    at the same company could never see the same document, and rag-api's
    tenants/{tenant}/members/{email} roster was bypassed entirely.

    The member doc is keyed by email so rag-api can do a point lookup, and
    carries `email` as a field so this reverse lookup is one collection-group
    query rather than a scan of every tenant.

    Not cached (12 September 2026). This answer sat under st.cache_data(ttl=300),
    so a person taken off the roster kept their tenant - and the upload right
    that comes with it (documents.py) - for up to five minutes on every instance
    that had answered them. The API, the chat service and the MCP server read
    the roster on every request (shared/tenancy.py); this image cannot import
    shared/ (its Dockerfile copies only this directory), so it does the same by
    hand. What is cached is the CONNECTION (_roster_db): one point query per
    rerun is cheap, a new client per rerun is not.
    """
    hits = (_roster_db().collection_group("members")
              .where("email", "==", email.lower()).limit(1).get())
    for doc in hits:
        return doc.reference.parent.parent.id      # tenants/{THIS}/members/{email}
    return None


@st.cache_resource(show_spinner=False)
def _roster_db():
    from google.cloud import firestore
    return firestore.Client()


def is_admin(user):
    email = user.get("email", "").lower()
    domain = email.split("@")[-1] if "@" in email else ""
    return email in ADMIN_EMAILS or domain in ADMIN_DOMAINS
'''
with open('auth.py', 'w') as f:
    f.write(AUTH_PY)
print('auth.py written')
print()
print('AUTH_MODE=iap uses IAP JWT verification (production)')
print('AUTH_MODE=oidc uses native st.login() (dev)')
print('ALWAYS verify JWT - never trust X-Goog-Authenticated-User-Email alone')


In [ ]:
CITATIONS_PY = '''
"""Citations on the screen - text, figures and video segments. Lesson 9.6, gap G7.

A citation is what the reader can OPEN. For a text chunk that is the page of the source; for a
figure it is the figure itself, inline, captioned by page; for a video segment it is the video,
started at the second the segment begins. The four extra fields (kind, media_url, start, end)
are optional and default to text, so a citation written before Module 9 renders exactly as it
did before Module 9 - and the pill in the prose says [Fig 3, p.12] or [Clip 2, 03:20] instead
of a bare [3], which is the M09 gate sentence made visible.
"""
import os, re
import datetime
import google.auth
import google.auth.credentials
import google.auth.transport.requests
import streamlit as st
from google.cloud import storage

_storage = storage.Client()
SIGNED_URL_EXPIRY = datetime.timedelta(minutes=15)
_CITE = re.compile(r"\\[(\\d+(?:\\s*,\\s*\\d+)*)\\]")

# SIGNING ON CLOUD RUN. The service's credential is a token from the metadata server, not a key, and
# generate_signed_url will not sign with it: "you need a private key to sign credentials" (the first
# live upload URL, 9 September 2026), with iam.serviceAccountTokenCreator granted and unused. Handed
# the service's email and access token instead, the library signs through the IAM signBlob API - the
# path that grant exists for. A credential that can sign itself (a key file, an impersonated
# credential in a notebook) needs nothing. services/rag-api/media.py carries the same twelve lines: the frontend image
# has no shared/ to import them from, and the gate holds the two copies to one shape.
def _signing_kwargs() -> dict:
    creds, _ = google.auth.default()
    if isinstance(creds, google.auth.credentials.Signing):
        return {}
    creds.refresh(google.auth.transport.requests.Request())
    return {"service_account_email": creds.service_account_email, "access_token": creds.token}


def signed_url(gcs_uri, page=None):
    # gcs_uri: gs://bucket/path -> V4 signed URL, signed through IAM as documind-ui-sa (sa.tf:
    # ui_self_impersonate). The first figure the lane cited would have thrown here without it.
    _, _, rest = gcs_uri.partition("gs://")
    bucket_name, _, blob_name = rest.partition("/")
    blob = _storage.bucket(bucket_name).blob(blob_name)
    url = blob.generate_signed_url(version="v4", expiration=SIGNED_URL_EXPIRY, method="GET", **_signing_kwargs())
    return f"{url}#page={page}" if page else url


def _mmss(seconds) -> str:
    try:
        s = int(float(seconds))
    except (TypeError, ValueError):
        return "?"
    return f"{s // 60:02d}:{s % 60:02d}"


def _label(n: int, src: dict | None) -> str:
    """What the pill says. Text: [3]. Figure: [Fig 3, p.12]. Segment: [Clip 3, 03:20]."""
    if not src:
        return f"[{n}]"
    kind = src.get("kind", "text")
    if kind in ("figure", "table"):
        page = src.get("page_start")
        return f"[{'Fig' if kind == 'figure' else 'Table'} {n}{f', p.{page}' if page else ''}]"
    if kind == "segment":
        return f"[Clip {n}, {_mmss(src.get('start'))}]"
    return f"[{n}]"


def render_with_citations(answer, sources):
    # sources: list of {"text", "source_uri", "page_start", "kind", "media_url", "start", "end"};
    # index N -> sources[N-1]. Only the first three are guaranteed; the rest default.
    def _pill(match):
        nums = [int(n) for n in match.group(1).replace(" ", "").split(",")]
        spans = []
        for n in nums:
            src = sources[n - 1] if 0 < n <= len(sources) else None
            tip = (src["text"][:120] + "...") if src else "unknown source"
            tip = tip.replace('"', '&quot;')
            spans.append(
                f'<span title="{tip}" style="background:#ccfbf1;color:#065f46;'
                f'border-radius:6px;padding:1px 7px;margin:0 2px;font-size:12px;'
                f'font-weight:600;cursor:help;">{_label(n, src)}</span>')
        return "".join(spans)
    html = _CITE.sub(_pill, answer)
    st.markdown(html, unsafe_allow_html=True)

    with st.expander(f"📚 Sources ({len(sources)})"):
        for i, src in enumerate(sources, 1):
            page = src.get("page_start")
            kind = src.get("kind", "text")
            st.markdown(f"**{_label(i, src)}** {src['text'][:200]}...")
            if src.get("effective_from"):
                st.caption(f"effective from {src['effective_from']}")     # the ledger (12.5): the document's date
            try:
                if kind in ("figure", "table") and src.get("media_url"):
                    # The figure itself, inline - retrieved BY its caption, shown TO the human.
                    st.image(signed_url(src["media_url"]),
                             caption=f"{_label(i, src)} · {src['source_uri'].rsplit('/', 1)[-1]}")
                elif kind == "segment" and src.get("media_url"):
                    start = int(float(src.get("start") or 0))
                    st.video(signed_url(src["media_url"]), start_time=start)
                    st.caption(f"{_mmss(src.get('start'))} – {_mmss(src.get('end'))} of "
                               f"{src['source_uri'].rsplit('/', 1)[-1]}")
                else:
                    url = signed_url(src["source_uri"], page=page)
                    label = f"Jump to page {page}" if page else "Open source"
                    st.link_button(label, url)
            except Exception as e:
                st.caption(f"(signed URL unavailable: {e})")
'''
with open('citations.py', 'w') as f:
    f.write(CITATIONS_PY)
print('citations.py written')


In [ ]:
ADMIN_PY = '''
"""The admin dashboard is a SEPARATE SERVICE. This module only links to it.

An earlier version of this file was a full second dashboard: its own BigQuery
queries, its own Firestore reads, its own tab layout - a duplicate of
services/admin/admin_dashboard.py with different code behind the same four
tab names.

The reason it is gone is not tidiness. Running those queries HERE runs them as
ui-sa, which has no bigquery.jobUser and no bigquery.dataViewer. Either the tab
fails, or somebody grants ui-sa those roles - and then an XSS in the chat page
reaches the warehouse, which is exactly the blast radius the three-service-
account split in 12.1 exists to prevent.
"""
import os

import streamlit as st

from auth import is_admin

ADMIN_URL = os.environ.get("ADMIN_URL", "")


def admin_page(user):
    # Defense in depth: hiding the nav entry is not access control.
    if not is_admin(user):
        st.error("403 - Admins only")
        st.stop()
    st.title("Admin")
    st.write("The DocuMind admin dashboard runs as its own Cloud Run service, "
             "under **admin-sa**, so that reporting credentials never live in "
             "the process that renders user chat.")
    if ADMIN_URL:
        st.link_button("Open the admin dashboard", ADMIN_URL)
    else:
        st.info("Set ADMIN_URL to the admin service's URL to enable this link.")
'''

with open('admin_dashboard.py', 'w') as f: f.write(ADMIN_PY)
print('admin_dashboard.py written - a link, not a second dashboard')


In [ ]:
CONFIG_TOML = '''
[server]
headless = true
address = "0.0.0.0"
port = 8080
# Both of these, or file uploads fail SILENTLY behind the load balancer:
# the LB rewrites Origin, Streamlit's CSRF check rejects the multipart POST,
# the spinner completes and no file arrives. Pairs with --session-affinity.
enableXsrfProtection = false
enableCORS = false
maxUploadSize = 200

[browser]
gatherUsageStats = false

[theme]
base = "light"
primaryColor = "#0d9488"
'''

import os
os.makedirs('.streamlit', exist_ok=True)
with open('.streamlit/config.toml', 'w') as f: f.write(CONFIG_TOML)
print('wrote .streamlit/config.toml')


In [ ]:
DOCKERFILE = '''
# syntax=docker/dockerfile:1.7
FROM python:3.12-slim

RUN apt-get update && apt-get install -y --no-install-recommends \\
      tini curl ca-certificates build-essential \\
    && rm -rf /var/lib/apt/lists/*

RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip \\
 && pip install --no-cache-dir -r requirements.txt

COPY --chown=app:app . .
USER app

ENV PORT=8080 \\
    STREAMLIT_SERVER_PORT=8080 \\
    STREAMLIT_SERVER_ADDRESS=0.0.0.0 \\
    STREAMLIT_SERVER_HEADLESS=true \\
    STREAMLIT_SERVER_ENABLE_CORS=false \\
    STREAMLIT_SERVER_ENABLE_XSRF_PROTECTION=false \\
    STREAMLIT_BROWSER_GATHER_USAGE_STATS=false \\
    PYTHONUNBUFFERED=1

EXPOSE 8080
HEALTHCHECK --interval=30s --timeout=5s --start-period=15s --retries=3 \\
    CMD curl -fsS http://localhost:8080/_stcore/health || exit 1

ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["streamlit","run","app.py", \\
     "--server.port=8080","--server.address=0.0.0.0", \\
     "--server.headless=true", \\
     "--server.enableCORS=false","--server.enableXsrfProtection=false"]
'''
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)

DEPLOY = '''
# IAP on the service itself (--iap): the run.app URL is the front door, IAP signs people in,
# and nothing reaches the container without an assertion. That takes --ingress=all (IAP
# fronts every path) and --no-allow-unauthenticated (IAP's own service agent is the only
# invoker, granted below). MIN_INSTANCES is 1 on a session day and 0 after.
#
# The env delimiter is ^|^: the values carry ':' (URLs) and ',' (ADMIN_EMAILS). RAG_API_URL
# is the API's deterministic run.app URL; CHAT_URL is empty in the lean profile and chat.py
# hides the surface; the VECTOR_ pair is empty there too. IAP_AUDIENCE is THIS service's:
# /projects/NUMBER/locations/REGION/services/documind-ui, project number, leading slash.
gcloud run deploy documind-ui \\
  --image=${REGION:-us-central1}-docker.pkg.dev/$PROJECT/documind/ui:$GIT_SHA \\
  --region=${REGION:-us-central1} --platform=managed \\
  --no-allow-unauthenticated --iap \\
  --ingress=all \\
  --memory=2Gi --cpu=2 --concurrency=80 --timeout=3600 \\
  --min-instances=${MIN_INSTANCES:-0} --max-instances=10 \\
  --cpu-boost --session-affinity --execution-environment=gen2 \\
  --service-account=documind-ui-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-env-vars="^|^AUTH_MODE=iap|RAG_API_URL=https://documind-api-$PROJECT_NUMBER.${REGION:-us-central1}.run.app|CHAT_URL=$CHAT_URL|SPEECH_REGION=asia-south1|GOOGLE_CLOUD_PROJECT=$PROJECT|GOOGLE_CLOUD_LOCATION=${REGION:-us-central1}|UPLOAD_BUCKET=$PROJECT-uploads|LAYOUT_PROCESSOR=projects/$PROJECT/locations/$DOCAI_LOCATION/processors/$DOCAI_PROCESSOR_ID|TTS_CACHE_BUCKET=$PROJECT-tts-cache|VECTOR_INDEX_ENDPOINT=$VECTOR_INDEX_ENDPOINT|DEPLOYED_INDEX_ID=$VECTOR_DEPLOYED_INDEX_ID|CHUNKS_COLLECTION=chunks|AUDIT_COLLECTION=audit_events|OBSERVABILITY_DATASET=documind_observability|RAG_MODEL=gemini-3.6-flash|ADMIN_EMAILS=$ADMIN_EMAILS|ADMIN_DOMAINS=$ADMIN_DOMAINS|IAP_AUDIENCE=/projects/$PROJECT_NUMBER/locations/${REGION:-us-central1}/services/documind-ui" \\
  --set-secrets="COOKIE_SECRET=cookie-secret:latest" \\
  --vpc-connector=projects/$PROJECT/locations/${REGION:-us-central1}/connectors/documind-vpc \\
  --vpc-egress=private-ranges-only

# Grant self-impersonation for V4 signed URLs
gcloud iam service-accounts add-iam-policy-binding \\
  documind-ui-sa@$PROJECT.iam.gserviceaccount.com \\
  --member="serviceAccount:documind-ui-sa@$PROJECT.iam.gserviceaccount.com" \\
  --role="roles/iam.serviceAccountTokenCreator"

# IAP's service agent is what invokes the service once IAP is on; gcloud does not grant it.
gcloud run services add-iam-policy-binding documind-ui \\
  --region=${REGION:-us-central1} --project=$PROJECT \\
  --member="serviceAccount:service-$PROJECT_NUMBER@gcp-sa-iap.iam.gserviceaccount.com" \\
  --role=roles/run.invoker

# Who may sign in: one grant per address in ADMIN_EMAILS (comma-separated). Signing in is
# not membership - the roster (make roster) decides which tenant each person sees.
for who in $(echo "$ADMIN_EMAILS" | tr ',' ' '); do
  gcloud iap web add-iam-policy-binding --project=$PROJECT \\
    --resource-type=cloud-run --service=documind-ui --region=${REGION:-us-central1} \\
    --member="user:$who" --role=roles/iap.httpsResourceAccessor
done
'''
print('Dockerfile written')
print(DEPLOY)
